# SlotSaver: No-Show Risk Model Evaluation & Explainability Analysis

> **Objective:** Evaluate Logistic Regression vs Gradient Boosting for appointment no-show prediction.
> **Core Requirement:** The business owner needs to understand and justify why a client is flagged for a deposit.

### Why Logistic Regression over Gradient Boosting?
1. **Owner Trust & Client Communication:** A black-box tree ensemble cannot cleanly explain to an upset customer why they must pay an upfront deposit. Logistic Regression provides exact odds ratios (e.g. *"Each prior no-show multiplies the odds of missing by 2.5x; booking >7 days out adds 40% risk"*).
2. **Calibration:** Logistic Regression produces well-calibrated probabilities $P(\text{no\_show}) \in [0, 1]$ directly optimizable for the deposit prompt threshold.
3. **Zero Latency:** In-memory dot-product evaluation completes in $<2\text{ms}$ during checkout.

In [1]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, roc_curve

model_path = Path("../model/model.joblib")
pipeline = joblib.load(model_path)
print("Loaded trained model pipeline successfully.")

In [2]:
from model.train import generate_synthetic_data

df_test = generate_synthetic_data(n_samples=2500, random_state=99)
feature_cols = [
    "lead_time_hours", "prior_no_shows", "prior_completed", "no_show_ratio",
    "day_of_week", "hour_of_day", "service_duration_min", "service_price", "deposit_paid"
]
X_test = df_test[feature_cols]
y_test = df_test["no_show"]

probs = pipeline.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, probs)
print(f"Holdout Test ROC-AUC: {auc:.4f}")

In [3]:
# Threshold Evaluation at 0.60
preds = (probs >= 0.60).astype(int)
print("Evaluation Report at Deposit Threshold (P >= 0.60):")
print(classification_report(y_test, preds))

In [4]:
# Standardized Coefficients
clf = pipeline.named_steps["classifier"]
coef_series = pd.Series(clf.coef_[0], index=feature_cols).sort_values(ascending=False)
print("Standardized Odds Multipliers (e^coef):")
for feat, coef in coef_series.items():
    odds_multiplier = np.exp(coef)
    print(f"{feat:<22}: coef={coef:>+.3f} | odds multiplier={odds_multiplier:.2f}x")